Data Ingestion

In [1]:
###Document Structure

from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="This is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Vineet Kumar",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Vineet Kumar', 'date_created': '2025-01-01'}, page_content='This is the main text content I am using to create RAG')

In [3]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

c:\Users\100916\Desktop\langchain\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
# Sample code for loading from Directory
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [7]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2025-10-08T17:32:48+05:30', 'source': '..\\data\\pdf\\Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'file_path': '..\\data\\pdf\\Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'total_pages': 53, 'format': 'PDF 1.5', 'title': 'Microsoft Word - Agreement For Sale-Grande 27012025', 'author': 'CAPriyesh', 'subject': '', 'keywords': '', 'moddate': '2025-10-08T17:32:48+05:30', 'trapped': '', 'modDate': "D:20251008173248+05'30'", 'creationDate': "D:20251008173248+05'30'", 'page': 0}, page_content='Page 1of53 \n \nAGREEMENT FOR SALE \n \nThis Agreement for Sale (“Agreement”) made and executed on this ____ day of \n________ in the year Two Thousand and Twenty-Five at Taluka Maval, District Pune, \nBETWEEN \nM/s IPRIMA INFRA LLP \n(LLPIN: ACG-4668) (PAN: AAKFI6643K),  \na Limited Liability Partnership Firm, \nHaving its office at: \nD-605+606, Business Court, Survey 

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf
  ✓ Loaded 53 pages

Processing: End_End_to_end_3rd_work .pdf
  ✓ Loaded 9 pages

Processing: Synergistic_Thickness_and_Density_for_High-Efficiency_FTO_SbSnO2_CsFAPbIBr3_BaSi2_Perovskite_Solar_Cells.pdf
  ✓ Loaded 4 pages

Total documents loaded: 66


In [11]:
###Text splitting and get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [12]:
chunks=split_documents(all_pdf_documents)
chunks

Split 66 documents into 221 chunks

Example chunk:
Content: Page 1of53  
AGREEMENT FOR SALE 
 
This Agreement for Sale (“Agreement”)  made and executed on this ____ day of 
________ in the year Two Thousand and Twenty-Five at Taluka Maval, District Pune, 
BETW...
Metadata: {'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2025-10-08T17:32:48+05:30', 'title': 'Microsoft Word - Agreement For Sale-Grande 27012025', 'author': 'CAPriyesh', 'moddate': '2025-10-08T17:32:48+05:30', 'source': '..\\data\\pdf\\Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': 'Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2025-10-08T17:32:48+05:30', 'title': 'Microsoft Word - Agreement For Sale-Grande 27012025', 'author': 'CAPriyesh', 'moddate': '2025-10-08T17:32:48+05:30', 'source': '..\\data\\pdf\\Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': 'Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'file_type': 'pdf'}, page_content='Page 1of53  \nAGREEMENT FOR SALE \n \nThis Agreement for Sale (“Agreement”)  made and executed on this ____ day of \n________ in the year Two Thousand and Twenty-Five at Taluka Maval, District Pune, \nBETWEEN \nM/s IPRIMA INFRA LLP \n(LLPIN: ACG-4668) (PAN: AAKFI6643K),  \na Limited Liability Partnership Firm, \nHaving its office at: \nD-605+606, Business Court, Survey No.707, \n Mukund Nagar, Pune 411 037 \nRepresented by its Designated Partner: \nMr GUNESH CHHUNILAL SANCHETI  \n(DPIN: 017

Embedding and Vector Store DB

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6110.34it/s]


Model loaded successfully. Embedding dimension: 384


VectorStore

In [15]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 442


In [16]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2025-10-08T17:32:48+05:30', 'title': 'Microsoft Word - Agreement For Sale-Grande 27012025', 'author': 'CAPriyesh', 'moddate': '2025-10-08T17:32:48+05:30', 'source': '..\\data\\pdf\\Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': 'Agreement To Sale - Mr. Vineet Kumar - Plot No. P62.pdf', 'file_type': 'pdf'}, page_content='Page 1of53  \nAGREEMENT FOR SALE \n \nThis Agreement for Sale (“Agreement”)  made and executed on this ____ day of \n________ in the year Two Thousand and Twenty-Five at Taluka Maval, District Pune, \nBETWEEN \nM/s IPRIMA INFRA LLP \n(LLPIN: ACG-4668) (PAN: AAKFI6643K),  \na Limited Liability Partnership Firm, \nHaving its office at: \nD-605+606, Business Court, Survey No.707, \n Mukund Nagar, Pune 411 037 \nRepresented by its Designated Partner: \nMr GUNESH CHHUNILAL SANCHETI  \n(DPIN: 017

In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 221 texts...


Batches: 100%|██████████| 7/7 [00:07<00:00,  1.14s/it]


Generated embeddings with shape: (221, 384)
Adding 221 documents to vector store...
Successfully added 221 documents to vector store
Total documents in collection: 663


Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [19]:
rag_retriever.retrieve("Writer independent (WI) module")

Retrieving documents for query: 'Writer independent (WI) module'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bb55b4a1_181',
  'content': 'independent ( WI) module with the baseline writer-dependent\n(WD) model leads to a notable improvement in overall system\nperformance. This enhancement can be attributed to the com-\nplementary strengths of both modules. The writer-dependent\ncomponent is adept at capturing fine-grained, writer-specific\nlocal features by training directly on fragments from the\ntarget writers. However, its ability to learn diverse patterns\nTABLE IV: Comparison of average identification rate (in %)\nusing different configuration of fusing writer dependent\n(WD) and writer independent ( WI) modules.\nIAM CVL CERUG-EN\nConfiguration Top 1 Top 5 Top 1 Top 5 Top 1 Top 5\nBaseline (WD) 92.07 96.43 89.76 95.46 96.05 99.56\nWI+WD+Max 93.13 96.51 90.58 96.25 97.08 100\nWI+WD+Add 93.43 97.35 91.12 96.79 97.18 100\nWI+WD+Concat 93.75 97.46 91.86 97.11 97.56 100\nis often limited, and constrained by the amount of text avail-\nable for training. However upon integration w

RAG Pipeline- VectorDB To LLM Output Generation

In [20]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [21]:
answer=rag_simple("What is llama?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is llama?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 131.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


pdf_path = r"C:\Users\100916\Desktop\langchain\data\pdf\End_End_to_end_3rd_work .pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile")

prompt = ChatPromptTemplate.from_template(
    "Summarize the following text:\n\n{text}"
)

chain = prompt | llm | StrOutputParser()

summaries = []

for chunk in chunks:
    result = chain.invoke({
        "text": chunk.page_content
    })
    summaries.append(result)

final_text = "\n".join(summaries)

final_summary = chain.invoke({
    "text": final_text
})

print(final_summary)

The text discusses various approaches to writer identification using handwriting fragments and neural networks. The main idea is to develop a system that can identify writers based on their handwriting, even with limited samples. The approaches include:

1. **Dual-stream convolutional neural networks (CNNs)**: These networks use two parallel streams to process handwriting fragments, one for writer-dependent features and one for writer-independent features.
2. **Attention mechanisms**: These mechanisms help focus on important regions of the handwriting fragments to improve writer identification.
3. **Fragment-based approach**: This approach involves processing individual fragments of handwriting, rather than whole words or texts.
4. **Multi-task learning**: This approach involves training a network to perform multiple tasks simultaneously, such as writer identification and handwriting recognition.

The text also discusses various experiments and results, including:

1. **Comparison of d